# Placement Prep Agent
## AI-Powered Job Readiness Assistant

### Hands-on project

We will build the project **module by module**.

**Module 1:** Interviewer Agent — generates mock interview questions
**Module 2:** Evaluator Agent — evaluates answers and reviews resumes
**Module 3:** Career Coach Agent — creates a company-specific study plan

We will complete and run one module before moving to the next.

**Final flow:** Company + Role → Research → Mock Interview → Evaluation → Study Plan → Application Tracking

## Teaching structure

For every module we will clearly identify:

**Input → Tool/Function → Processing → Output**

Then we connect that output to the next module.

> Resume review and application tracking are folded into the Evaluator and Memory sections respectively.

## Colab setup

For faster generation, select:

**Runtime → Change runtime type → T4 GPU**

CPU also works, but the local LLM will be slower.

In [1]:
!pip -q install -U transformers accelerate sentencepiece requests
print("✅ Installation complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
✅ Installation complete


# 1. Initialize the AI Model

The language model is the part of the system that understands instructions and generates text.

We use a small local instruction-following model so the project does not require a paid API key.

In [2]:
import json
import requests
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

Device: cpu
Running on CPU


In [3]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model_kwargs = {}
if device == "cuda":
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    **model_kwargs
).to(device)

model.eval()

print("✅ Model loaded:", MODEL_NAME)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [4]:
def ask_llm(user_prompt, system_prompt="You are a helpful placement preparation assistant that helps students prepare for job interviews."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

print("✅ LLM helper ready")

✅ LLM helper ready


# 🚀 MODULE 1 — Researching the Target Company

## Goal

Give the Agent a company name and obtain useful background information to ground the rest of the prep work.

### Input

```text
Company name
```

Example:

```text
Google
```

### Output

```text
Company research
```

This module teaches the first important Agent capability:

**Tool Integration**

## Module 1 — Step 1: Build the Research Tool

The research tool connects our Agent to an external information source.

Conceptually:

```text
Agent
  ↓
Research Tool
  ↓
External Information
  ↓
Research Result
```

In [5]:
def research_company(company):
    url = (
        "https://en.wikipedia.org/api/rest_v1/page/summary/"
        + requests.utils.quote(company)
    )

    try:
        response = requests.get(
            url,
            timeout=10,
            headers={"User-Agent": "PlacementPrepAgent/1.0"}
        )

        if response.status_code != 200:
            return {
                "company": company,
                "status": "not_found",
                "summary": f"No research result found for {company}.",
                "source": None
            }

        data = response.json()

        return {
            "company": company,
            "status": "success",
            "summary": data.get(
                "extract",
                "No company summary available."
            ),
            "source": data.get(
                "content_urls", {}
            ).get(
                "desktop", {}
            ).get(
                "page", ""
            )
        }

    except Exception as e:
        return {
            "company": company,
            "status": "error",
            "summary": f"Research failed: {e}",
            "source": None
        }

print("✅ Research tool created")

✅ Research tool created


## Module 1 — Step 2: Run the Module

Now we give the tool its input and inspect the output.

In [6]:
company = "Google"

research_result = research_company(company)

print("INPUT")
print("Company:", company)

print("\nOUTPUT")
print("Status:", research_result["status"])
print("Company:", research_result["company"])
print("\nResearch:")
print(research_result["summary"])
print("\nSource:")
print(research_result["source"])

INPUT
Company: Google

OUTPUT
Status: success
Company: Google

Research:
Google LLC is an American multinational technology corporation focused on information technology, online advertising, search engine technology, email, cloud computing, software, quantum computing, e-commerce, consumer electronics, and artificial intelligence (AI). It has been referred to as "the most powerful company in the world" by the BBC, and is one of the world's most valuable brands. Google's parent company Alphabet Inc. has been described as a Big Tech company.

Source:
https://en.wikipedia.org/wiki/Google


## Module 1 — Understand Input and Output

```text
INPUT
Google
   ↓
Research Tool
   ↓
OUTPUT
Company information
```

### What did we learn?

A tool gives the Agent a capability that the LLM does not have by itself — up-to-date, grounded company facts.

**Module 1 complete ✅**

# 🚀 MODULE 2 — Interviewer Agent (Mock Interview Generator)

Now we build the second module.

The important part is that we **reuse the output of Module 1**.

### Input

```text
Company + Role + Company Research
```

### Output

```text
Mock Interview Questions
```

Flow:

```text
Research Result
      ↓
Interviewer Agent
      ↓
Interview Questions
```

## Module 2 — Step 1: Build the Interviewer Tool

The LLM uses the company research as context so the questions are tailored instead of generic.

In [7]:
def generate_interview_questions(company, role, research):
    prompt = f"""
Act as an experienced technical interviewer preparing a mock interview.

Company:
{company}

Target role:
{role}

Company research:
{research}

Generate:
- 5 technical/role-specific questions
- 3 behavioral/HR questions

Requirements:
- Tailor at least two questions using details from the company research
- Order questions from easier to harder
- Number the questions clearly
- Do not invent company facts
"""

    return ask_llm(prompt)

print("✅ Interviewer tool created")

✅ Interviewer tool created


## Module 2 — Step 2: Run the Module

Notice that the input is coming from Module 1.

In [8]:
company = research_result["company"]
research = research_result["summary"]
role = "Software Engineer Intern"

interview_questions = generate_interview_questions(
    company,
    role,
    research
)

print("INPUT")
print("Company:", company)
print("Role:", role)
print("\nResearch supplied to the interviewer:")
print(research[:1500])

print("\nOUTPUT")
print("=" * 60)
print(interview_questions)

INPUT
Company: Google
Role: Software Engineer Intern

Research supplied to the interviewer:
Google LLC is an American multinational technology corporation focused on information technology, online advertising, search engine technology, email, cloud computing, software, quantum computing, e-commerce, consumer electronics, and artificial intelligence (AI). It has been referred to as "the most powerful company in the world" by the BBC, and is one of the world's most valuable brands. Google's parent company Alphabet Inc. has been described as a Big Tech company.

OUTPUT
### Technical Questions

1. **Question 1:**
   - **Description:** Given a piece of code with some bugs, how would you identify and fix them?
   - **Explanation:** This question tests your ability to debug code efficiently and understand the underlying logic of the program.

2. **Question 2:**
   - **Description:** Implement a function that calculates the nth Fibonacci number using recursion.
   - **Explanation:** Recursion 

## Module 2 — Understand Input and Output

```text
Module 1
Research Company
      ↓
Company Research
      ↓
Module 2
Interviewer Agent
      ↓
Mock Interview Questions
```

### What did we learn?

This is **workflow orchestration**: the result of one operation becomes input to another operation.

**Module 2 complete ✅**

# 🚀 MODULE 3 — Evaluator Agent (Answer Evaluation + Resume Review)

The third module reviews the candidate's spoken answers and their resume.

### Input

```text
Question + Candidate Answer
    or
Resume Text + Target Role
```

### Output

```text
Score + Feedback + Improvement Tips
```

The Evaluator gives the candidate honest, structured feedback rather than generic praise.

## Module 3 — Step 1: Build the Answer Evaluation Tool

In [9]:
def evaluate_answer(question, answer):
    prompt = f"""
Act as a strict but fair interview evaluator.

Interview question:
{question}

Candidate's answer:
{answer}

Provide:
1. A score out of 10
2. What was good about the answer
3. What was missing or weak
4. One concrete way to improve the answer

Be honest. Do not inflate the score.
"""

    return ask_llm(prompt)

print("✅ Answer evaluation tool created")

✅ Answer evaluation tool created


## Module 3 — Step 2: Build the Resume Review Tool

In [10]:
def review_resume(resume_text, target_role):
    prompt = f"""
Act as a resume reviewer for job placements.

Target role:
{target_role}

Resume text:
{resume_text}

Provide:
1. Three resume strengths
2. Three gaps or weaknesses relative to the target role
3. Three specific rewrite suggestions (bullet-level, not generic advice)

Keep the feedback practical and specific to the resume text given.
"""

    return ask_llm(prompt)

print("✅ Resume review tool created")

✅ Resume review tool created


## Module 3 — Step 3: Run the Module

In [ ]:
sample_question = "Tell me about a time you solved a difficult technical problem."
sample_answer = (
    "In my final year project, our IoT sensor kept losing connection. "
    "I debugged the firmware, found a power management bug, and fixed it "
    "by adjusting the sleep cycle. The project then worked reliably."
)

evaluation = evaluate_answer(sample_question, sample_answer)

print("INPUT")
print("Question:", sample_question)
print("Answer:", sample_answer)

print("\nOUTPUT — Answer Evaluation")
print("=" * 60)
print(evaluation)

In [ ]:
sample_resume = """
Computer Science student. Skills: Python, ESP32/IoT, deep learning, UI/UX,
web development. Built a speech synthesis hackathon project on speaker
identity preservation. Built a full-stack peer skill-exchange platform.
Emcee for college technical symposiums.
"""

resume_feedback = review_resume(sample_resume, role)

print("INPUT")
print("Target role:", role)
print("Resume text:", sample_resume.strip())

print("\nOUTPUT — Resume Review")
print("=" * 60)
print(resume_feedback)

## Module 3 — Understand Input and Output

```text
Candidate Answer / Resume
      ↓
Evaluator Agent
      ↓
Score + Feedback + Gaps
```

### What did we learn?

The Evaluator turns raw candidate output into **structured, actionable signal** — this signal is what the next module (Career Coach) will act on.

**Module 3 complete ✅**

# 🚀 MODULE 4 — Career Coach Agent (Company-Specific Study Plan)

The final module turns research + evaluator feedback into a concrete study plan.

### Input

```text
Company + Role + Company Research + Weak Areas
```

### Output

```text
Company-specific Study Plan
```

The plan will contain:
- Topics to prioritize, based on the company and role
- A week-by-week schedule
- Resources/practice types per topic
- A note on what to fix based on the Evaluator's feedback

## Module 4 — Step 1: Build the Study Plan Tool

In [ ]:
def prepare_study_plan(company, role, research, weak_areas):
    prompt = f"""
Act as a career coach creating a focused, company-specific study plan.

Company:
{company}

Target role:
{role}

Company research:
{research}

Candidate's weak areas (from interview/resume evaluation):
{weak_areas}

Create a 3-week study plan that includes:
1. Priority topics for this company and role
2. A week-by-week breakdown (Week 1, Week 2, Week 3)
3. Practice recommendations for each week (mock interviews, coding practice, projects)
4. How the plan specifically addresses the candidate's weak areas

Important:
- Do not invent company facts.
- Keep the plan practical and time-boxed.
"""

    return ask_llm(prompt)

print("✅ Study plan tool created")

## Module 4 — Step 2: Run the Module

In [ ]:
weak_areas = evaluation[:800]

study_plan = prepare_study_plan(
    company,
    role,
    research,
    weak_areas
)

print("INPUT")
print("Company:", company)
print("Role:", role)
print("\nWeak areas supplied to the Career Coach:")
print(weak_areas)

print("\nOUTPUT")
print("=" * 60)
print(study_plan)

## Module 4 — Understand Input and Output

```text
Company Research + Evaluator Feedback
      ↓
Career Coach Agent
      ↓
Company-Specific Study Plan
```

**Module 4 complete ✅**

# 🔗 Connect the Four Modules

We have already built and tested each module separately.

Now we connect them into one Placement Prep workflow.

```text
USER
  ↓
Company + Role + Resume + Sample Answer
  ↓
1. Research Company
  ↓
Company Research
  ↓
2. Interviewer Agent → Mock Interview Questions
  ↓
3. Evaluator Agent → Answer Evaluation + Resume Review
  ↓
4. Career Coach Agent → Study Plan
```

This is our complete hands-on application.

In [ ]:
def run_placement_prep_agent(company, role, resume_text, sample_answer, sample_question):
    print("PLACEMENT PREP AGENT")
    print("=" * 70)

    print("\n[1] Researching company...")
    research_result = research_company(company)

    if research_result["status"] != "success":
        print(research_result["summary"])
        return None

    research = research_result["summary"]
    print("✅ Research completed")

    print("\n[2] Generating mock interview questions...")
    questions = generate_interview_questions(company, role, research)
    print("✅ Interview questions generated")

    print("\n[3] Evaluating sample answer...")
    answer_eval = evaluate_answer(sample_question, sample_answer)
    print("✅ Answer evaluated")

    print("\n[4] Reviewing resume...")
    resume_review = review_resume(resume_text, role)
    print("✅ Resume reviewed")

    print("\n[5] Preparing company-specific study plan...")
    plan = prepare_study_plan(company, role, research, answer_eval[:800])
    print("✅ Study plan prepared")

    return {
        "company": company,
        "role": role,
        "research": research,
        "questions": questions,
        "answer_evaluation": answer_eval,
        "resume_review": resume_review,
        "study_plan": plan
    }

prep_result = run_placement_prep_agent(
    company="Google",
    role="Software Engineer Intern",
    resume_text=sample_resume,
    sample_answer=sample_answer,
    sample_question=sample_question
)

# 📊 Final Placement Prep Report

In [ ]:
if prep_result:
    print("=" * 70)
    print("FINAL PLACEMENT PREP REPORT")
    print("=" * 70)

    print("\n1. COMPANY RESEARCH")
    print("-" * 70)
    print(prep_result["research"])

    print("\n2. MOCK INTERVIEW QUESTIONS")
    print("-" * 70)
    print(prep_result["questions"])

    print("\n3. ANSWER EVALUATION")
    print("-" * 70)
    print(prep_result["answer_evaluation"])

    print("\n4. RESUME REVIEW")
    print("-" * 70)
    print(prep_result["resume_review"])

    print("\n5. COMPANY-SPECIFIC STUDY PLAN")
    print("-" * 70)
    print(prep_result["study_plan"])

# 📚 Connect the Project to Agentic AI Concepts

| Topic | Demonstration in our project |
|---|---|
| **AI Agents** | Placement Prep Agent |
| **Tool Integration** | Company research, evaluation, and study-plan tools |
| **Planning** | Choosing/organizing the required steps |
| **Workflow Orchestration** | Research → Interview → Evaluation → Study Plan |
| **Memory** | Application tracker storing progress per company |
| **Human-in-the-loop** | Candidate reviews/approves the study plan |
| **Agent Observability** | Track execution steps |
| **Multi-Agent Systems** | Split Interviewer, Evaluator, and Career Coach into specialized Agents |

The four hands-on modules remain the core of the project.

# 5. Memory — Application Tracker

Memory allows an Agent system to retain useful information between interactions — here, across every company a student applies to.

For this classroom project, we use a simple Python dictionary.

In [ ]:
agent_memory = {
    "user_preferences": {
        "target_roles": ["Software Engineer Intern"]
    },
    "applications": {}
}

agent_memory["applications"][prep_result["company"]] = {
    "role": prep_result["role"],
    "status": "prepping",
    "last_study_plan": prep_result["study_plan"][:1000],
    "resume_review": prep_result["resume_review"][:500]
}

print(json.dumps(agent_memory, indent=2)[:5000])

# 6. Human-in-the-loop

A human (the candidate) can review an AI-generated study plan before committing time to it.

This adds a safety and approval checkpoint.

In [ ]:
print("GENERATED STUDY PLAN")
print("=" * 60)
print(prep_result["study_plan"])

approval = input(
    "\nApprove this study plan and start following it? (yes/no): "
).strip().lower()

if approval == "yes":
    agent_memory["applications"][prep_result["company"]]["status"] = "plan_approved"
    print("✅ Candidate approved the study plan.")
else:
    agent_memory["applications"][prep_result["company"]]["status"] = "plan_rejected"
    print("🛑 Candidate rejected the study plan. Revise before using.")

# 7. Agent Observability

Observability means being able to see what happened during Agent execution.

A simple execution trace can record the tools and their status.

In [ ]:
execution_trace = [
    {"step": 1, "tool": "research_company", "status": "completed"},
    {"step": 2, "tool": "generate_interview_questions", "status": "completed"},
    {"step": 3, "tool": "evaluate_answer", "status": "completed"},
    {"step": 4, "tool": "review_resume", "status": "completed"},
    {"step": 5, "tool": "prepare_study_plan", "status": "completed"}
]

print("AGENT EXECUTION TRACE")
print("=" * 60)

for item in execution_trace:
    print(
        f"Step {item['step']}: "
        f"{item['tool']} → {item['status']}"
    )

# 8. Multi-Agent Systems

Our working implementation uses **one Placement Prep Agent with five tools**.

A multi-agent architecture could divide the responsibilities:

```text
                    Manager Agent
                         |
       +----------------+----------------+
       |                |                |
       ▼                ▼                ▼
 Interviewer        Evaluator       Career Coach
    Agent              Agent            Agent
```

Each specialized Agent has one responsibility, while the Manager Agent coordinates them and updates the Application Tracker (memory).

# 🎯 Final Project Summary

We built four required modules:

### 1. Researching the target company
**Input:** Company name
**Output:** Company research

### 2. Interviewer Agent — mock interview questions
**Input:** Company + Role + Research
**Output:** Tailored interview questions

### 3. Evaluator Agent — answer evaluation + resume review
**Input:** Question + Answer, or Resume + Role
**Output:** Score, feedback, gaps

### 4. Career Coach Agent — company-specific study plan
**Input:** Company + Role + Research + Weak areas
**Output:** Week-by-week study plan

### Complete architecture

```text
                 USER
                  |
                  ▼
       COMPANY + ROLE + RESUME
                  |
                  ▼
          ┌────────────────┐
          │ PLACEMENT PREP │
          │      AGENT     │
          └───────┬────────┘
                  |
                  ▼
        ┌────────────────────┐
        │ 1. RESEARCH TOOL   │
        └─────────┬──────────┘
                  |
                  ▼
           COMPANY RESEARCH
                  |
        ┌─────────┼─────────────┐
        ▼         ▼             ▼
 ┌────────────┐ ┌───────────┐ ┌──────────────┐
 │ 2. INTER-  │ │3. EVALUATOR│ │4. CAREER    │
 │  VIEWER    │ │   TOOL     │ │  COACH TOOL │
 └─────┬──────┘ └─────┬──────┘ └──────┬───────┘
       │              │               │
       ▼              ▼               ▼
  QUESTIONS   SCORE+FEEDBACK    STUDY PLAN
```

**One project → four modules → multiple Agent concepts.**